# Generic GLUE Quantized BERT Runner

Run MRPC, CoLA, RTE, SST-2, QQP, MNLI, or QNLI from this notebook by changing `TASK_NAME`.


In [ ]:
!pip install transformers==4.35.2
!pip install datasets evaluate fsspec


In [ ]:
import transformers
print(transformers.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys
from pathlib import Path

os.environ["HF_DATASETS_OFFLINE"] = "0"

PROJECT_DIR_PATH = globals().get("PROJECT_DIR_PATH", "/content/mrcp-tr-ptq")
PROJECT_DIR = Path(PROJECT_DIR_PATH)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}
!ls {PROJECT_DIR_PATH}


## Task And Config

Use `TASK_NAME = "mrpc"`, `TASK_NAME = "cola"`, `TASK_NAME = "rte"`, `TASK_NAME = "sst2"`, `TASK_NAME = "qqp"`, `TASK_NAME = "mnli"`, or `TASK_NAME = "qnli"`. The matching default config file is selected below.


In [ ]:
TASK_NAME = "mnli"  # "mrpc", "cola", "rte", "sst2", "qqp", "mnli", or "qnli"

CONFIG_BY_TASK = {
    "mrpc": "quant_config.json",
    "cola": "quant_config_cola.json",
    "rte": "quant_config_rte.json",
    "sst2": "quant_config_sst2.json",
    "qqp": "quant_config_qqp.json",
    "mnli": "quant_config_mnli.json",
    "qnli": "quant_config_qnli.json",
}

CONFIG_PATH = PROJECT_DIR / "notebooks" / "configs" / CONFIG_BY_TASK[TASK_NAME]
print("TASK_NAME:", TASK_NAME)
print("CONFIG_PATH:", CONFIG_PATH)


## Imports


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, BertConfig, BertTokenizerFast

from mrcp_quant import (
    apply_experiment_config,
    apply_layer_quant_overrides,
    get_task_spec,
    load_experiment_config,
    resolve_q_module_list,
    save_experiment_result,
)
from run_glue_quant import calibrate_model, evaluate_model, optimize_scale_factors, q_module_names, quantized_module_paths


## Model Initialization


In [ ]:
experiment_config = load_experiment_config(CONFIG_PATH)
experiment_config["task_name"] = TASK_NAME
apply_experiment_config(experiment_config)

task = get_task_spec(experiment_config.get("task_name", "mrpc"))
model_name = experiment_config.get("model_name", task.default_model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("task", task.name)
print("model_name", model_name)
print("device", device)

tokenizer = BertTokenizerFast.from_pretrained(model_name)
hf_config = BertConfig.from_pretrained(model_name)
hf_model = AutoModelForSequenceClassification.from_pretrained(model_name)

model = task.model_class(hf_config)
applied_layer_quant_overrides = apply_layer_quant_overrides(model, experiment_config)
if applied_layer_quant_overrides:
    print("Applied layer quantization overrides:", applied_layer_quant_overrides)

res = model.load_state_dict(hf_model.state_dict(), strict=False)
print("Missing keys:", len(res.missing_keys))
print("Unexpected keys:", len(res.unexpected_keys))
print("Missing examples:", res.missing_keys[:30])

model.to(device)


## Quantization Setup


In [ ]:
q_module_list = resolve_q_module_list(
    experiment_config.get("q_module_list", ["QLayerNorm"])
)

model.set_q_module_list(q_module_list)
model.set_quant()

note = []
for name, module in model.named_modules():
    q = getattr(module, "quant", None)
    opt = getattr(module, "is_opt_scale", None)
    if (q is True) or (opt is True):
        note.append((name, type(module).__name__, q, opt))

print("modules not in pure-float mode:", len(note))
print(*note[:50], sep="\n")


## Calibration And Scale Optimization


In [ ]:
calibrate_model(model, task, tokenizer, q_module_list, experiment_config, device)
optimize_scale_factors(model, task, tokenizer, q_module_list, experiment_config, device)


## Evaluation


In [ ]:
metrics, average_loss, num_examples = evaluate_model(
    model,
    task,
    tokenizer,
    experiment_config,
    device,
)

primary_metric_value = metrics[task.primary_metric]
print("metrics", metrics)
print(f"Final {task.primary_metric}:", primary_metric_value)
print("Final Loss:", average_loss)


## Save Result


In [ ]:
resolved_q_module_names = q_module_names(q_module_list)
output_config = dict(experiment_config)
output_config["q_module_list"] = resolved_q_module_names

result_path = save_experiment_result(
    accuracy=primary_metric_value,
    loss=average_loss,
    configuration=output_config,
    quantized=resolved_q_module_names,
    output_dir=PROJECT_DIR / "output",
    extra={
        "task_name": task.name,
        "model_name": model_name,
        "quantized_module_paths": quantized_module_paths(model),
        "metrics": metrics,
        "primary_metric_name": task.primary_metric,
        "primary_metric_value": primary_metric_value,
        "num_val_examples": num_examples,
    },
)
print("Saved results:", result_path)


In [ ]:
# import matplotlib.pyplot as plt

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 10,
# })

# labels = [
#     "TR-GELU",
#     "TR-Norm",
#     "TR-SoftMax"
# ]

# areas = [3024, 14352, 45025]

# total = sum(areas)

# legend_labels = [
#     rf"{name}"
#     for name, area in zip(labels, areas)
# ]

# fig, ax = plt.subplots(
#     figsize=(3.3, 3.6),
#     constrained_layout=True
# )

# wedges, texts, autotexts = ax.pie(
#     areas,
#     startangle=90,
#     autopct='%1.1f%%',
#     pctdistance=0.72,
#     wedgeprops=dict(edgecolor="white", linewidth=1.2),
#     textprops={"fontsize": 8}
# )

# ax.set_title("Post-synthesis Area Breakdown", fontsize=10)

# ax.legend(
#     wedges,
#     legend_labels,
#     loc="upper center",
#     bbox_to_anchor=(0.5, -0.02),
#     fontsize=8,
#     frameon=False,
#     ncol=3
# )

# plt.savefig("tr_area_breakdown.pdf", bbox_inches="tight")
# plt.savefig("tr_area_breakdown.png", dpi=300, bbox_inches="tight")

# plt.show()

In [ ]:
# import matplotlib.pyplot as plt

# plt.rcParams.update({
#     "font.family": "serif",
#     "font.size": 10,
# })

# labels = [
#     "Elementwise Mul",
#     "Softmax Logic",
#     "MAC / Vector Array",
#     "2nd Iteration",
#     "TR-ln"
# ]

# percentages = [
#     12.31,
#     4.96,
#     78.91,
#     1.83,
#     1.96
# ]

# fig, ax = plt.subplots(
#     figsize=(3.4, 3.6),
#     constrained_layout=True
# )

# wedges, texts, autotexts = ax.pie(
#     percentages,
#     startangle=90,
#     autopct='%1.1f%%',
#     pctdistance=0.72,
#     wedgeprops=dict(edgecolor="white", linewidth=1.2),
#     textprops={"fontsize": 8}
# )

# ax.set_title(
#     "Area Breakdown of 16-Element TR-SoftMax Engine",
#     fontsize=10
# )

# ax.legend(
#     wedges,
#     labels,
#     loc="upper center",
#     bbox_to_anchor=(0.5, -0.02),
#     fontsize=8,
#     frameon=False,
#     ncol=2
# )

# plt.savefig(
#     "tsoft_area_breakdown.pdf",
#     bbox_inches="tight"
# )

# plt.savefig(
#     "tsoft_area_breakdown.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# plt.show()